#### **DATA EXTRACTION:**

**Phase:** Data Understanding    
**Purpose:** Load all raw files exactly as downloaded, inspect their real structure,
document findings, and validate that `COUNTY_NAME_MAP` in `constants.py` covers every
county name variant in the files.

**What we now know about the real file structures:**
| File | County col | County values | Notes |
|---|---|---|---|
| Adult_on_ART.xlsx | `County` | `"Baringo County"` | Has ` County` suffix |
| Adult_on_HTS.xlsx | `County` | `"Baringo County"` | Same suffix as ART |
| VLT.xlsx | `County` | `"Baringo"` | Clean, no suffix |
| IIT.xlsx | `Region` | `"Central"`, `"Coast"` etc | **9 regions, not 47 counties** |
| individual_features.csv | `county` | integers 1–47 | Mapped via `DHS_COUNTY_MAP` |

All NSDCC Excel files have a **portal banner in row 0** — we use `header=1` to skip it.

#### Imports and Path Setup

In [13]:
import sys, os
sys.path.append(os.path.abspath('..'))   # point to repo root for constants.py

import pandas as pd
import numpy as np

from constants import (
    # File paths
    ADULT_ART_FILE, HTS_FILE, VLT_FILE, IIT_FILE, DHS_REDUCED,
    # Per-file column constants
    ART_COUNTY_COL, ART_PERIOD_COL, ART_COUNTY_SUFFIX,
    HTS_COUNTY_COL, HTS_PERIOD_COL, HTS_COUNTY_SUFFIX,
    VLT_COUNTY_COL, VLT_HAS_PERIOD,
    IIT_REGION_COL, IIT_HAS_PERIOD,
    # Maps
    COUNTY_NAME_MAP, DHS_COUNTY_MAP,
    IIT_REGION_MAP, REGION_TO_COUNTIES,
)

print('✓ Imports OK')
print(f'  ART  : {ADULT_ART_FILE}')
print(f'  HTS  : {HTS_FILE}')
print(f'  VLT  : {VLT_FILE}   (VLT_HAS_PERIOD={VLT_HAS_PERIOD})')
print(f'  IIT  : {IIT_FILE}   (IIT_HAS_PERIOD={IIT_HAS_PERIOD})')
print(f'  DHS  : {DHS_REDUCED}')

✓ Imports OK
  ART  : c:\Users\user\Desktop\MORINGA\Phase_5_Project\Phase-5-HIV-Care-Gap-AI\data\raw\Adult_on_ART.xlsx
  HTS  : c:\Users\user\Desktop\MORINGA\Phase_5_Project\Phase-5-HIV-Care-Gap-AI\data\raw\Adult_on_HTS.xlsx
  VLT  : c:\Users\user\Desktop\MORINGA\Phase_5_Project\Phase-5-HIV-Care-Gap-AI\data\raw\VLT.xlsx   (VLT_HAS_PERIOD=False)
  IIT  : c:\Users\user\Desktop\MORINGA\Phase_5_Project\Phase-5-HIV-Care-Gap-AI\data\raw\IIT.xlsx   (IIT_HAS_PERIOD=False)
  DHS  : c:\Users\user\Desktop\MORINGA\Phase_5_Project\Phase-5-HIV-Care-Gap-AI\data\raw\individual_features.csv


#### Load ART and HTS Files. (Both have Period Cplumn)



In [14]:
def load_with_period(filepath, period_col, label):
    """
    Load an NSDCC file that has BOTH County and Period columns (ART, HTS).
    Skips banner (header=1), drops Unnamed cols, drops rows where Period is NaN.
    """
    df = pd.read_excel(filepath, header=1, dtype=str)
    df = df.loc[:, ~df.columns.str.startswith('Unnamed')]
    df = df[df[period_col].notna()].reset_index(drop=True)
    rows, cols = df.shape
    print(f'  {label:<6} {rows:>3} rows x {cols:>3} cols  |  missing: {df.isnull().sum().sum()}')
    return df

print('Loading ART and HTS (both have Period column):')
art_raw = load_with_period(ADULT_ART_FILE, ART_PERIOD_COL, 'ART')
hts_raw = load_with_period(HTS_FILE,       HTS_PERIOD_COL, 'HTS')
print('✓ Loaded')

Loading ART and HTS (both have Period column):
  ART     47 rows x  11 cols  |  missing: 0
  HTS     47 rows x   7 cols  |  missing: 0
✓ Loaded


#### Load VLT (No Period column-single snapshot of 47 counties)

In [15]:
def load_snapshot(filepath, county_col, label):
    """
    Load a file with County but NO Period column (VLT).
    Skips banner (header=1), drops Unnamed cols.
    Drops rows where county column is NaN.
    """
    df = pd.read_excel(filepath, header=1, dtype=str)
    df = df.loc[:, ~df.columns.str.startswith('Unnamed')]
    df = df[df[county_col].notna()].reset_index(drop=True)
    rows, cols = df.shape
    print(f'  {label:<6} {rows:>3} rows x {cols:>3} cols  |  missing: {df.isnull().sum().sum()}')
    print(f'         ⚠  No Period column — single-period snapshot')
    return df

print('Loading VLT (no Period column):')
vlt_raw = load_snapshot(VLT_FILE, VLT_COUNTY_COL, 'VLT')
print()
print('VLT columns:', list(vlt_raw.columns))
vlt_raw.head(3)

Loading VLT (no Period column):
  VLT     47 rows x   7 cols  |  missing: 2
         ⚠  No Period column — single-period snapshot

VLT columns: ['County', 'Valid VL <15yrs', 'Suppressed <15yrs', 'Valid VL 15+Male', 'Suppressed 15+Male', 'Valid VL 15+Female', 'Suppressed 15+Female']


,County,Valid VL <15yrs,Suppressed <15yrs,Valid VL 15+Male,Suppressed 15+Male,Valid VL 15+Female,Suppressed 15+Female
0,Baringo,73,56,807,743,1821,1701
1,Bomet,272,230,2276,2112,5230,4872
2,Bungoma,326,289,4039,3848,10589,10221


#### Load IIT (No Period column just 9 regions of the country)

In [16]:
def load_iit(filepath, region_col, label):
    """
    Load the IIT file: Region column (not County), no Period column.
    Expects 9 rows: 8 MOH regions + 1 national Kenya total.
    """
    df = pd.read_excel(filepath, header=1, dtype=str)
    df = df.loc[:, ~df.columns.str.startswith('Unnamed')]
    df = df[df[region_col].notna()].reset_index(drop=True)
    rows, cols = df.shape
    print(f'  {label:<6} {rows:>3} rows x {cols:>3} cols  |  missing: {df.isnull().sum().sum()}')
    print(f'         ⚠  Region-level only, no Period column')
    return df

print('Loading IIT (Region column, no Period):')
iit_raw = load_iit(IIT_FILE, IIT_REGION_COL, 'IIT')
print()
print('IIT columns:', list(iit_raw.columns))
print()
print(f'Unique "{IIT_REGION_COL}" values:')
for r in iit_raw[IIT_REGION_COL].dropna().unique():
    std = IIT_REGION_MAP.get(r, '⚠ NOT IN MAP')
    n   = len(REGION_TO_COUNTIES.get(std, []))
    print(f'  {r:<20} → standardised: {str(std):<15}  ({n} counties)')
print()
iit_raw

Loading IIT (Region column, no Period):
  IIT      9 rows x  13 cols  |  missing: 6
         ⚠  Region-level only, no Period column

IIT columns: ['Region', 'Actively on Treatment by the Beginning of 2025 Children', 'Children IIT', 'Actively on Treatment by the Beginning of 2025 female adult', 'Female adult IIT', 'Actively on Treatment by the Beginning of 2025 Male', 'Male adult IIT', 'All Adult(Male & Female) on Treament at the begining of 2025', 'All Adult(Male & Female) IIT', 'All Adult(Male & Female) IIT percentage(%)', 'Children IIT(%)', 'Male adult IIT(%)', 'Female adult IIT(%)']

Unique "Region" values:
  Rift Valley          → standardised: Rift Valley      (14 counties)
  Western              → standardised: Western          (4 counties)
  Eastern              → standardised: Eastern          (8 counties)
  Nyanza               → standardised: Nyanza           (6 counties)
  Central              → standardised: Central          (5 counties)
  Coast                → standardise

,Region,Actively on Treatment by the Beginning of 2025 Children,Children IIT,Actively on Treatment by the Beginning of 2025 female adult,Female adult IIT,Actively on Treatment by the Beginning of 2025 Male,Male adult IIT,All Adult(Male & Female) on Treament at the begining of 2025,All Adult(Male & Female) IIT,All Adult(Male & Female) IIT percentage(%),Children IIT(%),Male adult IIT(%),Female adult IIT(%)
0,Rift Valley,474,62,7339,1125,3920,598,11259,1723,0.153,0.131,0.153,0.153
1,Western,271,19,3730,341,1703,145,5433,486,0.0895,0.07,0.085,0.091
2,Eastern,241,17,3336,310,1745,141,5081,451,0.0888,0.071,0.081,0.09300000000000001
3,Nyanza,607,52,9045,812,4415,417,13460,1229,0.0913,0.086,0.094,0.09
4,Central,142,12,2690,259,1422,155,4112,414,0.1007,0.085,0.109,0.096
5,Coast,161,26,2592,244,1199,107,3791,351,0.0926,0.161,0.08900000000000001,0.094
6,Nairobi,169,18,4898,564,2494,271,7392,835,0.113,0.107,0.109,0.115
7,North Eastern,NaN,NaN,NaN,NaN,NaN,NaN,0,0,-,-,-,-
8,Total,2065,206,33630,3655,16898,1834,50528,5489,0.10859999999999999,0.1,0.109,0.109


#### Loading DHS Individual_features.csv

In [17]:
dhs_raw = pd.read_csv(DHS_REDUCED, low_memory=False)
print(f'DHS individual_features.csv')
print(f'  Shape   : {dhs_raw.shape[0]:,} rows x {dhs_raw.shape[1]} cols')
print(f'  Columns : {list(dhs_raw.columns)}')
print(f'  Missing : {dhs_raw.isnull().sum().sum():,}')
print()
dhs_raw.head(3)

DHS individual_features.csv
  Shape   : 32,156 rows x 15 cols
  Columns : ['case_id', 'county', 'age_group', 'education_level', 'wealth_index', 'worked_last_12months', 'ever_tested_hiv', 'tested_hiv_last_12months', 'distance_to_facility', 'marital_status', 'currently_in_union', 'num_sexual_partners', 'knows_aids_death', 'told_hiv_positive', 'has_health_insurance']
  Missing : 146,760



,case_id,county,age_group,education_level,wealth_index,worked_last_12months,ever_tested_hiv,tested_hiv_last_12months,distance_to_facility,marital_status,currently_in_union,num_sexual_partners,knows_aids_death,told_hiv_positive,has_health_insurance
0,1 4 2,1,4,0,4,0,1.0,1.0,10.0,1,1,0.0,NaN,0.0,NaN
1,1 7 2,1,5,2,5,1,NaN,NaN,NaN,1,1,NaN,NaN,NaN,NaN
2,1 10 1,1,4,1,5,2,1.0,1.0,6.0,4,2,1.0,NaN,0.0,NaN


#### Inspect ART column structure

In [18]:
print('=== ART (Adult_on_ART.xlsx) ===')
print(f'Columns ({len(art_raw.columns)}):')
for c in art_raw.columns:
    print(f'  {c}')
print()
print(f'Period values : {sorted(art_raw[ART_PERIOD_COL].dropna().unique())}')
print(f'County sample : {sorted(art_raw[ART_COUNTY_COL].dropna().unique()[:4])}')
art_raw.head(3)

=== ART (Adult_on_ART.xlsx) ===
Columns (11):
  Period
  County
  MOH 731_HIV_TB_OnART_15-19_(F)_HV03-24
  MOH 731_HIV_TB_OnART_15-19_(M)_HV03-23
  MOH 731_HIV_TB_OnART_20-24_(F)_HV03-26
  MOH 731_HIV_TB_OnART_20-24_(M)_HV03-25
  MOH 731_HIV_TB_OnART_25+_(F)_HV03-28
  MOH 731_HIV_TB_OnART_25+_(M)_HV03-27
  Total
  Total_Males
  Total_Females

Period values : ['December 2025']
County sample : ['Baringo County', 'Bomet County', 'Bungoma County', 'Busia County']


,Period,County,MOH 731_HIV_TB_OnART_15-19_(F)_HV03-24,MOH 731_HIV_TB_OnART_15-19_(M)_HV03-23,MOH 731_HIV_TB_OnART_20-24_(F)_HV03-26,MOH 731_HIV_TB_OnART_20-24_(M)_HV03-25,MOH 731_HIV_TB_OnART_25+_(F)_HV03-28,MOH 731_HIV_TB_OnART_25+_(M)_HV03-27,Total,Total_Males,Total_Females
0,December 2025,Baringo County,82,70,164,92,3629,1701,5738,1863,3875
1,December 2025,Bomet County,234,160,416,175,7745,3616,12346,3951,8395
2,December 2025,Bungoma County,591,450,1235,400,20203,7717,30596,8567,22029


#### Inspect HTS Column Structure

In [19]:
print('=== HTS (Adult_on_HTS.xlsx) ===')
print(f'Columns ({len(hts_raw.columns)}):')
for c in hts_raw.columns:
    print(f'  {c}')
print()
print(f'Period values : {sorted(hts_raw[HTS_PERIOD_COL].dropna().unique())}')
print(f'County sample : {sorted(hts_raw[HTS_COUNTY_COL].dropna().unique()[:4])}')
hts_raw.head(3)

=== HTS (Adult_on_HTS.xlsx) ===
Columns (7):
  Period
  County
  MOH 731_HTS_Tests _(F) (Including PMTCT)_ HV01-02
  MOH 731_HTS_Tests _(M)_ HV01-01
  Total
  Total_Males
  Total_Females

Period values : ['2025']
County sample : ['Baringo County', 'Bomet County', 'Bungoma County', 'Busia County']


,Period,County,MOH 731_HTS_Tests _(F) (Including PMTCT)_ HV01-02,MOH 731_HTS_Tests _(M)_ HV01-01,Total,Total_Males,Total_Females
0,2025,Baringo County,49731,11697,61428,11697,49731
1,2025,Bomet County,82677,17389,100066,17389,82677
2,2025,Bungoma County,197275,47767,245042,47767,197275


#### Inspect VLT column structure

- VLT is a **single snapshot**  47 rows, one per county, no Period.  
- In the merge step, VLT columns are broadcast onto every period row via a county-only join.

In [20]:
print('VLT (VLT.xlsx) — single snapshot, NO Period')
print(f'Columns ({len(vlt_raw.columns)}):')
for c in vlt_raw.columns:
    print(f'  {c}')
print()
print(f'Has Period column : {"Period" in vlt_raw.columns}   (VLT_HAS_PERIOD={VLT_HAS_PERIOD})')
print(f'County sample     : {sorted(vlt_raw[VLT_COUNTY_COL].dropna().unique()[:5])}')
vlt_raw.head(5)

VLT (VLT.xlsx) — single snapshot, NO Period
Columns (7):
  County
  Valid VL <15yrs
  Suppressed <15yrs
  Valid VL 15+Male
  Suppressed 15+Male
  Valid VL 15+Female
  Suppressed 15+Female

Has Period column : False   (VLT_HAS_PERIOD=False)
County sample     : ['Baringo', 'Bomet', 'Bungoma', 'Busia', 'Elgeyo Marakwet']


,County,Valid VL <15yrs,Suppressed <15yrs,Valid VL 15+Male,Suppressed 15+Male,Valid VL 15+Female,Suppressed 15+Female
0,Baringo,73,56,807,743,1821,1701
1,Bomet,272,230,2276,2112,5230,4872
2,Bungoma,326,289,4039,3848,10589,10221
3,Busia,247,218,5024,4818,10792,10454
4,Elgeyo Marakwet,112,92,986,939,2332,2211


#### Inspect IIT column structure

- IIT is a **single-period region snapshot** 9 rows, no Period, no County.  
- The Kenya national total row is dropped in notebook 02.  
- Each region is then expanded to its constituent counties using `REGION_TO_COUNTIES`.

In [21]:
print('=== IIT (IIT.xlsx) — 9 region rows, NO Period, NO County ===')
print(f'Columns ({len(iit_raw.columns)}):')
for c in iit_raw.columns:
    print(f'  {c}')
print()
print(f'Has Period column : {"Period" in iit_raw.columns}   (IIT_HAS_PERIOD={IIT_HAS_PERIOD})')
print(f'Has County column : {"County" in iit_raw.columns}')
print(f'Has Region column : {IIT_REGION_COL in iit_raw.columns}')
iit_raw

=== IIT (IIT.xlsx) — 9 region rows, NO Period, NO County ===
Columns (13):
  Region
  Actively on Treatment by the Beginning of 2025 Children
  Children IIT
  Actively on Treatment by the Beginning of 2025 female adult
  Female adult IIT
  Actively on Treatment by the Beginning of 2025 Male
  Male adult IIT
  All Adult(Male & Female) on Treament at the begining of 2025
  All Adult(Male & Female) IIT
  All Adult(Male & Female) IIT percentage(%)
  Children IIT(%)
  Male adult IIT(%)
  Female adult IIT(%)

Has Period column : False   (IIT_HAS_PERIOD=False)
Has County column : False
Has Region column : True


,Region,Actively on Treatment by the Beginning of 2025 Children,Children IIT,Actively on Treatment by the Beginning of 2025 female adult,Female adult IIT,Actively on Treatment by the Beginning of 2025 Male,Male adult IIT,All Adult(Male & Female) on Treament at the begining of 2025,All Adult(Male & Female) IIT,All Adult(Male & Female) IIT percentage(%),Children IIT(%),Male adult IIT(%),Female adult IIT(%)
0,Rift Valley,474,62,7339,1125,3920,598,11259,1723,0.153,0.131,0.153,0.153
1,Western,271,19,3730,341,1703,145,5433,486,0.0895,0.07,0.085,0.091
2,Eastern,241,17,3336,310,1745,141,5081,451,0.0888,0.071,0.081,0.09300000000000001
3,Nyanza,607,52,9045,812,4415,417,13460,1229,0.0913,0.086,0.094,0.09
4,Central,142,12,2690,259,1422,155,4112,414,0.1007,0.085,0.109,0.096
5,Coast,161,26,2592,244,1199,107,3791,351,0.0926,0.161,0.08900000000000001,0.094
6,Nairobi,169,18,4898,564,2494,271,7392,835,0.113,0.107,0.109,0.115
7,North Eastern,NaN,NaN,NaN,NaN,NaN,NaN,0,0,-,-,-,-
8,Total,2065,206,33630,3655,16898,1834,50528,5489,0.10859999999999999,0.1,0.109,0.109


#### **SUMMARY TABLE:**

In [22]:
print(f"{'File':<25} {'Rows':>5} {'Cols':>5}  {'Period':>8}  {'Col'}  {'Format'}")
print('-' * 80)
print(f"{'Adult_on_ART.xlsx':<25} {art_raw.shape[0]:>5} {art_raw.shape[1]:>5}  {'Yes':>8}  County  'Baringo County' (strip suffix)")
print(f"{'Adult_on_HTS.xlsx':<25} {hts_raw.shape[0]:>5} {hts_raw.shape[1]:>5}  {'Yes':>8}  County  'Baringo County' (strip suffix)")
print(f"{'VLT.xlsx':<25} {vlt_raw.shape[0]:>5} {vlt_raw.shape[1]:>5}  {'No':>8}  County  'Baringo' (clean, no suffix)")
print(f"{'IIT.xlsx':<25} {iit_raw.shape[0]:>5} {iit_raw.shape[1]:>5}  {'No':>8}  Region  9 regions → expand to 47 counties")
print(f"{'individual_features.csv':<25} {dhs_raw.shape[0]:>5} {dhs_raw.shape[1]:>5}  {'N/A':>8}  county  integers 1-47 → DHS_COUNTY_MAP")


File                       Rows  Cols    Period  Col  Format
--------------------------------------------------------------------------------
Adult_on_ART.xlsx            47    11       Yes  County  'Baringo County' (strip suffix)
Adult_on_HTS.xlsx            47     7       Yes  County  'Baringo County' (strip suffix)
VLT.xlsx                     47     7        No  County  'Baringo' (clean, no suffix)
IIT.xlsx                      9    13        No  Region  9 regions → expand to 47 counties
individual_features.csv   32156    15       N/A  county  integers 1-47 → DHS_COUNTY_MAP


#### Validate all maps against actual file values

In [23]:
import importlib, constants
importlib.reload(constants)
from constants import COUNTY_NAME_MAP, IIT_REGION_MAP, REGION_TO_COUNTIES, DHS_COUNTY_MAP

def validate_county_map(raw_values, label, strip_suffix=None):
    check = [v.replace(strip_suffix, '').strip() for v in raw_values] if strip_suffix else list(raw_values)
    unmapped = [n for n in check if n not in COUNTY_NAME_MAP]
    if unmapped:
        print(f'⚠  {label} — {len(unmapped)} NOT in COUNTY_NAME_MAP:')
        for n in sorted(unmapped):
            print(f"     add '{n}': '{n}'")
    else:
        print(f'✓  {label} — all {len(raw_values)} values covered')

# ART and HTS: strip suffix before checking
validate_county_map(art_raw[ART_COUNTY_COL].dropna().unique(), 'ART', strip_suffix=ART_COUNTY_SUFFIX)
validate_county_map(hts_raw[HTS_COUNTY_COL].dropna().unique(), 'HTS', strip_suffix=HTS_COUNTY_SUFFIX)

# VLT: plain names, no stripping
validate_county_map(vlt_raw[VLT_COUNTY_COL].dropna().unique(), 'VLT')

# IIT: validate region names against IIT_REGION_MAP
print()
iit_regions  = iit_raw[IIT_REGION_COL].dropna().unique()
unmapped_reg = [r for r in iit_regions if r not in IIT_REGION_MAP]
if unmapped_reg:
    print(f'⚠  IIT — {len(unmapped_reg)} regions NOT in IIT_REGION_MAP:')
    for r in unmapped_reg:
        print(f"     add '{r}' to IIT_REGION_MAP")
else:
    print(f'✓  IIT — all {len(iit_regions)} region values covered by IIT_REGION_MAP')

# DHS: integer codes
print()
dhs_codes = dhs_raw['county'].dropna().astype(float).astype(int).unique()
unmapped_codes = [c for c in dhs_codes if c not in DHS_COUNTY_MAP]
if unmapped_codes:
    print(f'⚠  DHS — codes NOT in DHS_COUNTY_MAP: {unmapped_codes}')
else:
    print(f'✓  DHS — all {len(dhs_codes)} county codes covered')

✓  ART — all 47 values covered
✓  HTS — all 47 values covered
✓  VLT — all 47 values covered

✓  IIT — all 9 region values covered by IIT_REGION_MAP

✓  DHS — all 47 county codes covered


#### Validate REGION_TO_COUNTIES covers all 47 counties

In [24]:
all_in_regions   = [c for cs in REGION_TO_COUNTIES.values() for c in cs]
all_standardised = set(COUNTY_NAME_MAP.values())

print(f'REGION_TO_COUNTIES: {len(all_in_regions)} counties across {len(REGION_TO_COUNTIES)} regions')
print(f'COUNTY_NAME_MAP   : {len(all_standardised)} standardised county names')
print()

missing = all_standardised - set(all_in_regions)
extra   = set(all_in_regions) - all_standardised
dups    = [c for c in all_in_regions if all_in_regions.count(c) > 1]

print('✓' if not missing else '⚠',
      f'Missing from regions: {sorted(missing)}' if missing else 'All 47 counties present in REGION_TO_COUNTIES')
print('✓' if not extra else '⚠',
      f'Extra in regions: {sorted(extra)}' if extra else 'No extra county names in REGION_TO_COUNTIES')
print('✓' if not dups else '⚠',
      f'Duplicates: {sorted(set(dups))}' if dups else 'No duplicate counties')

REGION_TO_COUNTIES: 47 counties across 8 regions
COUNTY_NAME_MAP   : 47 standardised county names

✓ All 47 counties present in REGION_TO_COUNTIES
✓ No extra county names in REGION_TO_COUNTIES
✓ No duplicate counties
